<a href="https://colab.research.google.com/github/waelydz/ASD-Sense/blob/main/Model_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install matplotlib
!pip install kagglehub
!pip install scikit-learn seaborn

In [ ]:
import os
# Disable XLA and autotuning for unsupported new architectures
os.environ['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [ ]:
import tensorflow as tf

# Check if TensorFlow sees the GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Success! TensorFlow found {len(gpus)} GPU(s):")
    for gpu in gpus:
        print(gpu)
else:
    print("Warning: No GPU detected. TensorFlow is using the CPU.")

Success! TensorFlow found 1 GPU(s):
PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


Setup and Data Download

In [ ]:
import os
import shutil
import tensorflow as tf
import matplotlib.pyplot as plt
import kagglehub

DATASET_DIR = kagglehub.dataset_download('imranliaqat32/autism-spectrum-disorder-in-childrenhandgestures')
BINARY_DIR = os.path.join(DATASET_DIR, "binary_format_v2")

if not os.path.exists(BINARY_DIR):
    os.makedirs(os.path.join(BINARY_DIR, "yes_ASD"), exist_ok=True)
    os.makedirs(os.path.join(BINARY_DIR, "no_ASD"), exist_ok=True)

    for root, dirs, files in os.walk(DATASET_DIR):
        if "binary_format" in root:
            continue

        for file_name in files:
            if file_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                src_path = os.path.join(root, file_name)
                if "Non-ASD" in root:
                    shutil.copy(src_path, os.path.join(BINARY_DIR, "no_ASD", file_name))
                elif "ASD" in root:
                    shutil.copy(src_path, os.path.join(BINARY_DIR, "yes_ASD", file_name))

BATCH_SIZE = 32
IMG_SIZE = (224, 224)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    BINARY_DIR,
    label_mode='binary',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    BINARY_DIR,
    label_mode='binary',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_dataset.class_names
num_classes = len(class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)

Using Colab cache for faster access to the 'autism-spectrum-disorder-in-childrenhandgestures' dataset.


OSError: [Errno 30] Read-only file system: '/kaggle/input/autism-spectrum-disorder-in-childrenhandgestures/binary_format_v2'

Model Architecture

In [ ]:
from tensorflow.keras import layers, models

def build_custom_cnn(num_classes):
    model = models.Sequential([
        layers.InputLayer(shape=(224, 224, 3)),
        layers.Rescaling(1./255),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

cnn_model = build_custom_cnn(num_classes)

Compilation

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=['accuracy'],
    jit_compile=False
)

Training

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint("best_custom_cnn.keras", save_best_only=True)

cnn_history = cnn_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[checkpoint]
)

In [ ]:
# Load the best saved model weights
best_cnn = tf.keras.models.load_model("best_custom_cnn.keras")

# Run an official evaluation on the validation set
print("\nEvaluating the Best Saved Model...")
val_loss, val_acc = best_cnn.evaluate(val_dataset)

print("-" * 30)
print(f"OFFICIAL FINAL ACCURACY: {val_acc * 100:.2f}%")
print(f"OFFICIAL FINAL LOSS:     {val_loss:.4f}")
print("-" * 30)

Plotting

In [ ]:

# Find the epoch with the lowest validation loss
best_epoch = cnn_history.history['val_loss'].index(min(cnn_history.history['val_loss']))

plt.figure(figsize=(12, 4))

# --- Accuracy Plot ---
plt.subplot(1, 2, 1)
plt.plot(cnn_history.history['accuracy'], label='Training Accuracy')
plt.plot(cnn_history.history['val_accuracy'], label='Validation Accuracy')
plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Weights Saved (Epoch {best_epoch + 1})')
plt.title('Custom CNN Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# --- Loss Plot ---
plt.subplot(1, 2, 2)
plt.plot(cnn_history.history['loss'], label='Training Loss')
plt.plot(cnn_history.history['val_loss'], label='Validation Loss')
plt.axvline(x=best_epoch, color='r', linestyle='--', label=f'Best Weights Saved (Epoch {best_epoch + 1})')
plt.title('Custom CNN Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig("cnn_training_metrics_updated.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

def generate_evaluation_metrics(model, dataset, model_name):
    print(f"Analyzing {model_name}...")
    y_true = []
    y_pred_probs = []

    # Iterate through the dataset to extract exact labels and predictions
    for images, labels in dataset:
        y_true.extend(labels.numpy().flatten())
        preds = model.predict_on_batch(images)
        y_pred_probs.extend(preds.flatten())

    y_true = np.array(y_true)
    # Convert probability percentages to strict 0 or 1 predictions
    y_pred = (np.array(y_pred_probs) > 0.5).astype(int)

    # --- 1. Plot Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    # Using a clean blue color map for academic reporting
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-ASD', 'ASD'],
                yticklabels=['Non-ASD', 'ASD'],
                annot_kws={"size": 16})
    plt.title(f'{model_name} Confusion Matrix', fontsize=14, pad=15)
    plt.ylabel('Actual Diagnosis', fontsize=12)
    plt.xlabel('Predicted Diagnosis', fontsize=12)
    plt.tight_layout()

    # Automatically save a high-res copy for your Overleaf document
    file_name = f"{model_name.lower().replace(' ', '_')}_matrix.png"
    plt.savefig(file_name, dpi=300)
    plt.show()

    # --- 2. Print Classification Report ---
    print(f"\nDetailed Metrics for {model_name}:")
    print(classification_report(y_true, y_pred, target_names=['Non-ASD', 'ASD']))

generate_evaluation_metrics(best_cnn, val_dataset, "Custom CNN")